# Deep LDA Covariance Summary

Build a table of mean final accuracies from `deep_lda_covariances.json`.


In [1]:
import json
from pathlib import Path

import pandas as pd

json_path = Path('deep_lda_dnll.json')
with json_path.open('r', encoding='utf-8') as f:
    data = json.load(f)


In [2]:
results = data['results']
rows = []
for dataset_name, covs in results.items():
    for cov_type, runs in covs.items():
        final_test = [r['final_test_acc'] for r in runs]
        mean_test = sum(final_test) / len(final_test)
        variance = sum((x - mean_test) ** 2 for x in final_test) / len(final_test)
        std_test = variance ** 0.5
        rows.append({
            'dataset': dataset_name,
            'covariance': cov_type,
            'mean_test': mean_test,
            'std_test': std_test,
        })

df = pd.DataFrame(rows)
cov_order = ['spherical', 'diag', 'full']
dataset_order = ['FashionMNIST', 'CIFAR10', 'CIFAR100']
df['covariance'] = pd.Categorical(df['covariance'], categories=cov_order, ordered=True)
df['dataset'] = pd.Categorical(df['dataset'], categories=dataset_order, ordered=True)
df['summary'] = df.apply(lambda r: f"{r['mean_test']*100:.2f} \u00b1 {r['std_test']*100:.2f}", axis=1)
table = df.pivot(index='covariance', columns='dataset', values='summary')
table = table.reindex(index=cov_order, columns=dataset_order)
table


dataset,FashionMNIST,CIFAR10,CIFAR100
covariance,,,
spherical,91.92 ± 1.92,90.24 ± 0.32,67.40 ± 0.42
diag,NaN,NaN,NaN
full,NaN,NaN,NaN


## CIFAR-100 Accuracy Curves

Mean training and test accuracy across runs for each covariance type.


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'legend.fontsize': 14,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
})

def build_mean_curve(dataset_name):
    runs_by_cov = data['results'][dataset_name]
    rows = []
    for cov_type, runs in runs_by_cov.items():
        for run in runs:
            for m in run['epoch_metrics']:
                rows.append({
                    'covariance': cov_type,
                    'epoch': m['epoch'],
                    'train_acc': m['train_acc'],
                    'test_acc': m['test_acc'],
                })
    curve_df = pd.DataFrame(rows)
    return (
        curve_df
        .groupby(['covariance', 'epoch'])[['train_acc', 'test_acc']]
        .mean()
        .reset_index()
    )

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), sharey=True, constrained_layout=True)
datasets = ['CIFAR10', 'CIFAR100']
cov_order = ['spherical', 'diag', 'full']
colors = {
    'spherical': '#1f77b4',
    'diag': '#ff7f0e',
    'full': '#2ca02c',
}
linestyles = {'train': '-', 'test': '--'}

for ax, dataset_name in zip(axes, datasets):
    mean_curve = build_mean_curve(dataset_name)
    for cov_type in cov_order:
        subset = mean_curve[mean_curve['covariance'] == cov_type]
        ax.plot(
            subset['epoch'],
            subset['train_acc'] * 100,
            color=colors[cov_type],
            linestyle=linestyles['train'],
            label=f'{cov_type} train',
        )
        ax.plot(
            subset['epoch'],
            subset['test_acc'] * 100,
            color=colors[cov_type],
            linestyle=linestyles['test'],
            label=f'{cov_type} test',
        )
    ax.set_title(f'{dataset_name} Accuracy Curves')
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Accuracy (%)')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='upper left',
    bbox_to_anchor=(0.225, -.02),
    frameon=True,
    ncol=3,
)

plt.savefig('deep_qda_covariances_curves.png', dpi=600, bbox_inches='tight', pad_inches=0.1)
